In [1]:
import os, yaml, sys
import numpy as np
import matplotlib.pyplot as plt
import h5py
from IPython.display import clear_output
import random
from scipy.io import loadmat
ENV = os.getenv("MY_ENV", "dev")
with open("../../config.yaml", "r") as f:
    config = yaml.safe_load(f)
paths = config[ENV]["paths"]
sys.path.append(paths["src_path"])
sys.path.append(paths["useful_stuff_path"])
from useful_stuff.general_utils.utils import get_lagplot
# # from useful_stuff.general_utils.RSA import  dRSA
# # from useful_stuff.general_utils.II import  dRSA, dynInformationImbalance
from useful_stuff.image_processing.computational_models import get_relevant_output_layers
from project_specific_utils.dataloader import load_concat_regressout_meg
# from project_specific_utils.subsampling_lagged_comparisons import multivariate_lagged_comparisons
from image_processing.gaze_dep_models import save_ANN_features


In [2]:
from dataclasses import dataclass, field

@dataclass
class Cfg:
    sub_num = 3
    neu_fs = 100
    gaze_fs = 50
    sq_side = 384
    sensors_group = 'occ'
    model_name = "dino_v3_l"
    pkg = "hf"
    pseudotrial_len = 600
    pseudotrials_n = 100
    full_model_name = "alexnet_features.4"
    n_model_components = 1000
    pooling = "all"
    PCs_to_regress_out = 50
    timepts_to_regress_out = (-100, 100)
    iterations_n = 10
    repetition = 0
    signal_metric = "correlation"
    model_metric = "correlation"
    regress_out_gaze = 'PCR' # or None or pointwise
    PCs_to_regress_out = 50
    analysis_type = "RSA"
cfg = Cfg()

layers = get_relevant_output_layers(cfg.model_name)
repetitions = [0,1]
subjects = config["subjects"]
mod_fs = config["movie_fs"]
model_len = [round(i*cfg.neu_fs/config["movie_fs"]) for i in config["model_len"]]

In [31]:
from project_specific_utils.dataloader import load_eyetracking_data
from useful_stuff.general_utils.utils import TimeSeries, print_wise


In [62]:
from project_specific_utils.dataloader import load_meg_data
from useful_stuff.general_utils.regression import dyn_linear_encoding

def load_concat_regressout_meg_(paths, sub_num, repetition, sensors_group, neu_fs, gaze_fs, regress_out_gaze,  PCs_to_regress_out, timepts_to_regress_out=(-100,100), rank=0):
    neu = []
    print_wise(f"Loading MEG signal: {regress_out_gaze=}", rank)
    runs = np.arange(1,4)+3*repetition 
    nans_rows = []
    model_len = [round(length*neu_fs/config["movie_fs"]) for length in config["model_len"]]
    for idx, i_run in enumerate(runs):
        run_neu, labels = load_meg_data(paths, sub_num, i_run, sensors_group, neu_fs)
        if np.any(np.isnan(run_neu)):
            print_wise(i_run, "contains nan")
        run_neu.z_score_feats()
        run_neu = TimeSeries(run_neu[:model_len[idx]], run_neu.get_fs())
        # checks if there is any sensor that has been discarded
        nan_positions = np.argwhere(np.isnan(run_neu.array))
        if nan_positions.size > 0:
            print(f"Total NaNs: {nan_positions.shape[0]}")
        rows_with_nan = np.unique(nan_positions[:, 0])
        if rows_with_nan.size > 0:
            for bad_row in rows_with_nan:
                if np.all(np.isnan(run_neu.array[bad_row, :])):  
                    nans_rows.append(bad_row)
                else:
                    raise ValueError(f"Not all elements in {bad_row} are NaNs")
                # end
        if idx == 1: # i.e. if it is the 2nd part of the movie (run 2 or run 5)
            run_neu = run_neu[3*neu_fs:]
        neu.append(run_neu[:])
    # end for i_run in range(1,4):
    for idx, run in enumerate(neu):
        neu[idx] = np.delete(run, nans_rows, axis=0)

    for idx, run_neu in enumerate(neu):
        if regress_out_gaze:
            run_gaze, _ = load_eyetracking_data(paths, sub_num, idx +1+ repetition*3, gaze_fs, xy=True)
            run_gaze.z_score_feats()
            run_gaze.resample(run_gaze.get_fs())
            run_gaze = TimeSeries(run_gaze[:model_len[idx]], run_gaze.get_fs())
            run_neu = TimeSeries(run_neu, neu_fs)
            dyn_regr_obj = dyn_linear_encoding('lr', 'same', None)
            if regress_out_gaze == "PCR":
                run_neu, _ = dyn_regr_obj.delay_embed_PCR_regress_out(run_gaze, run_neu, timepts_to_regress_out, PCs_to_keep=PCs_to_regress_out, pad_mode='edge', crop_end=True)
            elif regress_out_gaze == "lag0":
                run_neu = dyn_regr_obj.pointwise_regress_out(run_gaze, run_neu, regression_type=None) 
            # end if cfg.regr_out_eyes == "PCR":
        # for some subjects there is a minimal difference of timing (the gaze is 1 datapoint shorter than the model)
        if idx == 1:
            timepts_diff = np.abs(len(run_neu)-(model_len[idx]- 3*neu_fs)) 
        else:
            timepts_diff = np.abs(len(run_neu)-model_len[idx]) 
        if timepts_diff == 0:
            pass # nothing to check
        elif timepts_diff > 0 and timepts_diff < 2:
            # the syntax of pad wants a tuple of tuples - ((rows_to_pad_at_st, rows_to_pad_at_end), (cols_to_pad_at_st, cols_to_pad_at_end))
            print_wise(f"Padding {timepts_diff} timepoints to the neural signal to match model length")
            run_neu = np.pad(run_neu.array, ((0, 0), (0, timepts_diff)), mode='edge') # edge means that we are repeating the edge values
        elif timepts_diff > 2:
            raise IndexError(
                f"The difference in length between the model and the regress_out neural signal is too big ({timepts_diff})"
            )
        neu[idx] = run_neu[:]
    # end if cfg.regr_out_eyes:
    len_runs = [i.shape for i in neu]
    print_wise(f"Shape runs {runs}: {len_runs}", rank=rank)
    neu = np.concatenate(neu, axis=1)
    neu = TimeSeries(neu, neu_fs)
    return neu
# EOF



# 13:40:20 - rank 0 Namespace(analysis_type='RSA', sub_num=11, sensors_group='occ', repetition=1, pseudotrial_len=600, pseudotrials_n=100, iterations_n=1000, neu_fs=100, gaze_fs=50, regress_out_gaze='PCR', PCs_to_regress_out=50, timepts_to_regress_out=(-100, 100), model_name='dino_v3_l', n_model_components=1000, sq_side=384, pkg='hf', pooling='all', signal_metric='cosine_cnt', model_metric='cosine_cnt')
# 13:40:20 - rank 0 sent 1 to 1
# 13:40:20 - rank 0 sent 2 to 2
# 13:40:20 - rank 0 sent 3 to 3
# 13:40:23 - rank 2 Shape runs [4 5 6]: [(41, 87379), (41, 86278), (41, 79070)]
# 13:40:23 - rank 2 received: 1
# 13:40:23 - rank 0 Loading model dino_v3_l_layer.1.mlp.down_proj: regress_out_gaze=False
# 13:40:23 - rank 3 Shape runs [4 5 6]: [(41, 87379), (41, 86278), (41, 79070)]
# 13:40:23 - rank 3 received: 2
# 13:40:23 - rank 0 Loading model dino_v3_l_layer.2.mlp.down_proj: regress_out_gaze=False
# 13:40:24 - rank 1 Shape runs [4 5 6]: [(41, 87379), (41, 86278), (41, 79070)]
# 13:40:24 - rank 1 received: 0
# 13:40:24 - rank 0 Loading model dino_v3_l_layer.0.mlp.down_proj: regress_out_gaze=False
# 13:40:24 - rank 2 dino_v3_l_layer.1.mlp.down_proj: shape runs [4 5 6]: [(1000, 87379), (1000, 86278), (1000, 79071)]
# 13:40:24 - rank 3 dino_v3_l_layer.2.mlp.down_proj: shape runs [4 5 6]: [(1000, 87379), (1000, 86278), (1000, 79071)]
# Rank 2 failed with error: X and Y TimeSeries must have the same length, got 252727 and 252728. 
# --------------------------------------------------------------------------
# MPI_ABORT was invoked on rank 2 in communicator MPI_COMM_WORLD
#   Proc: [[7100,1],2]
#   Errorcode: 1

# NOTE: invoking MPI_ABORT causes Open MPI to kill all MPI processes.
# You may or may not see output from other processes, depending on
# exactly when Open MPI kills them.
# --------------------------------------------------------------------------

In [3]:
from project_specific_utils.dataloader import load_concat_gaze
for i_sub in [9,]:
    for i_rep in repetitions[1:]:
        n_old = load_concat_regressout_meg(paths, i_sub, i_rep, cfg.sensors_group, cfg.neu_fs, cfg.gaze_fs, cfg.regress_out_gaze, cfg.PCs_to_regress_out, timepts_to_regress_out=cfg.timepts_to_regress_out, rank=0)
        # load_concat_gaze(paths, i_sub, i_rep, 50, 100, rank=0)

19:06:28 - rank 0 Loading MEG signal: regress_out_gaze='PCR'
19:06:32 - Padding 1 timepoints to the neural signal to match model length
19:06:32 - rank 0 Shape runs [4 5 6]: [(41, 87379), (41, 86278), (41, 79071)]


In [34]:
for l in layers:
    full_model_name = f"{cfg.model_name}_{l}"
    m = load_concat_regressout_mod(paths, 11, save_ANN_features, full_model_name, 1, mod_fs, cfg.neu_fs, *(cfg.sq_side, cfg.n_model_components, cfg.pooling), regress_out_gaze=False, gaze_dep=True, gaze_fs=50, rank=0,)


18:17:19 - rank 0 Loading model dino_v3_l_layer.0.mlp.down_proj: regress_out_gaze=False
18:17:20 - rank 0 dino_v3_l_layer.0.mlp.down_proj: shape runs [4 5 6]: [(1000, 87379), (1000, 86278), (1000, 79071)]
18:17:21 - rank 0 Loading model dino_v3_l_layer.1.mlp.down_proj: regress_out_gaze=False
18:17:21 - rank 0 dino_v3_l_layer.1.mlp.down_proj: shape runs [4 5 6]: [(1000, 87379), (1000, 86278), (1000, 79071)]
18:17:22 - rank 0 Loading model dino_v3_l_layer.2.mlp.down_proj: regress_out_gaze=False
18:17:23 - rank 0 dino_v3_l_layer.2.mlp.down_proj: shape runs [4 5 6]: [(1000, 87379), (1000, 86278), (1000, 79071)]
18:17:24 - rank 0 Loading model dino_v3_l_layer.3.mlp.down_proj: regress_out_gaze=False
18:17:24 - rank 0 dino_v3_l_layer.3.mlp.down_proj: shape runs [4 5 6]: [(1000, 87379), (1000, 86278), (1000, 79071)]
18:17:26 - rank 0 Loading model dino_v3_l_layer.4.mlp.down_proj: regress_out_gaze=False
18:17:26 - rank 0 dino_v3_l_layer.4.mlp.down_proj: shape runs [4 5 6]: [(1000, 87379), (1000